# AutoGluon Experiment: AutoML / Neural Network / Hybrid Model

Standalone experiment -- does **not** touch `main.py` / `notebooks/pipeline.ipynb`.

The course brief lists neural networks (MLP, transformers) and hybrid models
(text embeddings + tree/regression models) as example approaches we haven't
actually tried yet -- everything so far has been linear models, gradient
descent from scratch, and gradient-boosted trees.

**What we attempted and why it's scoped the way it is:** AutoGluon's deep
multimodal text model (`AG_TEXT_NN`, a real transformer) requires an NVIDIA
GPU internally (via `pynvml`) even when asked to run on CPU -- a hard
dependency in this AutoGluon version that Apple Silicon (MPS, no CUDA) can't
satisfy. That's an upstream library limitation, not something we can fix from
here. Instead, this notebook runs AutoGluon's standard (non-multimodal) model
zoo, which still includes a genuine neural network
(`NeuralNetTorch`/`NeuralNetFastAI`, both MLPs) alongside CatBoost, Random
Forest, Extra Trees, LightGBM, XGBoost, and a weighted ensemble -- and lets
AutoGluon auto-handle the raw text columns (`description`, `amenities`,
`bathrooms_text`) and categoricals itself, with no manual encoding.

We feed it the richest feature set we have (numeric + raw categoricals + raw
text + aspect sentiment) and compare against our best manual result so far
(XGBoost, 66.00% R^2 with aspect sentiment).


## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import CLEANED_TARGET_PATH, RANDOM_STATE, RAW_LISTINGS_PATH, TARGET_COLUMN, TEST_SIZE
from src.data.clean import clean_missing_target
from src.data.load import load_reviews
from src.evaluate import print_metrics_log_target
from src.features.build_features import build_feature_matrix
from src.features.categorical import fit_categorical_encoders, transform_categorical_features
from src.features.sentiment import compute_aspect_sentiment
from src.models.tree_models import train_xgboost

from autogluon.tabular import TabularPredictor


## Step 1: Build features (numeric + categorical + raw text kept) + aspect sentiment

In [2]:
clean_missing_target(RAW_LISTINGS_PATH, CLEANED_TARGET_PATH, target_column=TARGET_COLUMN)
df = pd.read_csv(CLEANED_TARGET_PATH)
df = build_feature_matrix(df)

reviews = load_reviews()
aspect_sentiment = compute_aspect_sentiment(reviews)
aspect_score_cols = [c for c in aspect_sentiment.columns if c != "listing_id"]

df = df.merge(aspect_sentiment, left_on="id", right_on="listing_id", how="left").drop(columns=["listing_id"])
df[aspect_score_cols] = df[aspect_score_cols].fillna(0)
df.shape


Reading /Users/jnanadeep/Desktop/airbnb-price-prediction/data/raw/listings.csv...



--- Processing Summary ---
Initial row count:          40769
Rows with missing 'price': 953 (Removed)
Remaining row count:        39816
Cleaned data saved to:      /Users/jnanadeep/Desktop/airbnb-price-prediction/data/processed/airbnb_rio_cleaned_target.csv



Dropping columns with >90.0% missing values:
 - neighborhood_overview
 - host_since
 - host_response_time
 - host_response_rate
 - host_acceptance_rate
 - host_thumbnail_url
 - host_neighbourhood
 - host_total_listings_count
 - host_verifications
 - neighbourhood
 - neighbourhood_group_cleansed
 - calendar_updated
 - license
 - instant_bookable

Imputed 25 numerical columns using their median value.
--- Running Feature Selection ---
Original unique columns: 76
Filtered columns retained: 24
--- Running Dynamic Outlier Removal ---
Calculated 99.0th percentile threshold: $6,999.13
Removed 399 listing(s) outside the range [$10 - $6,999.13].
New maximum price in dataset: $6,990.00


(39417, 32)

## Step 2: Shared train/test split (same split used throughout the other experiments)

In [3]:
df_model = df.drop(columns=["id"])
df_model["price_log"] = np.log1p(df_model[TARGET_COLUMN])
df_model = df_model.drop(columns=[TARGET_COLUMN])

train_df, test_df = train_test_split(df_model, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print("Columns AutoGluon will see:", list(df_model.columns))


Train: (31533, 31), Test: (7884, 31)
Columns AutoGluon will see: ['latitude', 'longitude', 'neighbourhood_cleansed', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'host_is_superhost', 'host_listings_count', 'hosts_time_as_host_years', 'minimum_nights', 'maximum_nights', 'availability_365', 'has_availability', 'number_of_reviews', 'review_scores_rating', 'reviews_per_month', 'description', 'dist_to_beach', 'amenity_count', 'desc_word_count', 'host_quality_score', 'cleanliness_score', 'location_safety_score', 'amenities_score', 'total_reviews_count', 'price_log']


## Step 3: Fit AutoGluon (full model zoo: LightGBM, XGBoost, CatBoost, Random Forest,
Extra Trees, NeuralNetTorch, NeuralNetFastAI, KNN, + weighted ensemble)

`num_gpus=0` avoids the pynvml/NVIDIA GPU-detection crash entirely -- this
path doesn't touch the multimodal text model, so it isn't affected by that bug.

In [4]:
predictor = TabularPredictor(label="price_log", problem_type="regression", verbosity=2).fit(
    train_data=train_df,
    time_limit=600,
    num_gpus=0,
)
predictor.leaderboard(test_df, silent=True)


No path specified. Models will be saved in: "AutogluonModels/ag-20260708_162928"


Verbosity: 2 (Standard Logging)


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.9
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       9.87 GB / 24.00 GB (41.1%)
Disk Space Avail:   776.32 GB / 926.30 GB (83.8%)


No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, TabDPT, and TabM. Requires a GPU and `pip install autogluon.tabular[tabarena]` to install TabPFN, TabICL, and TabDPT.
	presets='best'     : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='best_v150': New in v1.5: Better quality than 'best' and 5x+ faster to train. Give it a try!
	presets='high'     : Strong accuracy with fast inference speed.
	presets='high_v150': New in v1.5: Better quality than 'high' and 5x+ faster to train. Give it a try!


Using hyperparameters preset: hyperparameters='default'


...

	0.0s	 = Training   runtime


	0.0s	 = Validation runtime


AutoGluon training complete, total runtime = 830.89s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 10943.7 rows/s (2501 batch size)


TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/Users/jnanadeep/Desktop/airbnb-price-prediction/notebooks/AutogluonModels/ag-20260708_162928")


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.413142,-0.420138,root_mean_squared_error,0.805080,0.228534,816.016004,0.014612,0.000128,0.002728,2,True,4
1,LightGBM,-0.416892,-0.426408,root_mean_squared_error,0.248046,0.076655,19.623101,0.248046,0.076655,19.623101,1,True,2
2,LightGBMXT,-0.419262,-0.424051,root_mean_squared_error,0.235638,0.084979,17.086125,0.235638,0.084979,17.086125,1,True,1
3,RandomForestMSE,-0.445695,-0.450861,root_mean_squared_error,0.306784,0.066772,779.304050,0.306784,0.066772,779.304050,1,True,3


## Step 4: Evaluate AutoGluon's best model on the held-out test set (same metric convention as the rest of the project: R^2 in log-space, MAE/RMSE in real dollars)

In [5]:
y_test_log = test_df["price_log"]
autogluon_preds_log = predictor.predict(test_df.drop(columns=["price_log"]))
autogluon_metrics = print_metrics_log_target("AutoGluon (best model)", y_test_log, autogluon_preds_log)


AutoGluon (best model)         | R^2 Score: 68.61% | MAE: $234.40 | RMSE: $480.39


## Step 5: Compare against our best manual result (XGBoost + categorical encoding + aspect sentiment)

In [6]:
# numeric_cols already includes the aspect sentiment score columns, since they were
# merged into df in Step 1 before this selection runs -- no need to add them again.
numeric_cols = df.drop(columns=[TARGET_COLUMN, "id"], errors="ignore").select_dtypes(include=[np.number]).columns

y_log = np.log1p(df[TARGET_COLUMN])
train_manual, test_manual, y_train_log, y_test_log_manual = train_test_split(
    df, y_log, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

encoders = fit_categorical_encoders(train_manual)
X_train = pd.concat([train_manual[numeric_cols], transform_categorical_features(train_manual, encoders)], axis=1)
X_test = pd.concat([test_manual[numeric_cols], transform_categorical_features(test_manual, encoders)], axis=1)

xgb_model = train_xgboost(X_train, y_train_log)
xgb_preds_log = xgb_model.predict(X_test)
xgb_metrics = print_metrics_log_target("Manual XGBoost (categorical + aspect sentiment)", y_test_log_manual, xgb_preds_log)


Manual XGBoost (categorical + aspect sentiment) | R^2 Score: 66.00% | MAE: $245.14 | RMSE: $498.81


## Step 6: Summary

In [7]:
comparison = pd.DataFrame({
    "AutoGluon (best model)": autogluon_metrics,
    "Manual XGBoost (categorical + aspect sentiment)": xgb_metrics,
}).T[["mae", "rmse", "r2"]]
comparison


,mae,rmse,r2
AutoGluon (best model),234.396073,480.389527,0.686144
Manual XGBoost (categorical + aspect sentiment),245.138150,498.810873,0.659979


## Step 7: Which model did AutoGluon pick as best, and did the neural nets compete?

In [8]:
predictor.leaderboard(test_df, silent=True)


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.413142,-0.420138,root_mean_squared_error,0.747697,0.228534,816.016004,0.016269,0.000128,0.002728,2,True,4
1,LightGBM,-0.416892,-0.426408,root_mean_squared_error,0.230390,0.076655,19.623101,0.230390,0.076655,19.623101,1,True,2
2,LightGBMXT,-0.419262,-0.424051,root_mean_squared_error,0.222536,0.084979,17.086125,0.222536,0.084979,17.086125,1,True,1
3,RandomForestMSE,-0.445695,-0.450861,root_mean_squared_error,0.278502,0.066772,779.304050,0.278502,0.066772,779.304050,1,True,3


## Step 8: Force a GPU-accelerated neural net into the ensemble

The unrestricted run above only fit LightGBM/LightGBMXT/RandomForest within the
600s budget -- RandomForest alone took 774s and starved out CatBoost, XGBoost,
ExtraTrees, and both neural nets before they got a turn. No neural network was
actually trained, so point 2 ("try a neural network") wasn't really completed.

`autogluon.tabular`'s `NeuralNetTorch` model has an explicit Apple Silicon (MPS)
code path (`autogluon/tabular/models/tabular_nn/torch/tabular_nn_torch.py`,
`_get_device`): if `num_gpus=1` is passed and no CUDA device is found, it
selects `torch.device("mps")` and trains on the Mac's own GPU. This is a
different code path from the multimodal/`AG_TEXT_NN` text transformer that
crashed earlier (that one hard-requires `pynvml`, i.e. an NVIDIA GPU, even to
just log GPU info -- an upstream limitation on Apple Silicon). Verified
directly: `next(model.model.parameters()).device` reports `mps:0` after fit.

This run explicitly restricts hyperparameters to `GBM` + `XGB` + `NN_TORCH`
(dropping RandomForest, CatBoost, ExtraTrees, KNN, FastAI) so the neural net
actually gets a real share of the time budget, with `num_gpus=1` routing
`NeuralNetTorch` to MPS.

In [9]:
predictor_gpu = TabularPredictor(label="price_log", problem_type="regression", verbosity=2).fit(
    train_data=train_df,
    hyperparameters={"GBM": {}, "XGB": {}, "NN_TORCH": {}},
    time_limit=300,
    num_gpus=1,
)
predictor_gpu.leaderboard(test_df, silent=True)


No path specified. Models will be saved in: "AutogluonModels/ag-20260708_164325"


Verbosity: 2 (Standard Logging)


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.9
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:39:29 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       9.83 GB / 24.00 GB (41.0%)
Disk Space Avail:   769.67 GB / 926.30 GB (83.1%)


No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, TabDPT, and TabM. Requires a GPU and `pip install autogluon.tabular[tabarena]` to install TabPFN, TabICL, and TabDPT.
	presets='best'     : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='best_v150': New in v1.5: Better quality than 'best' and 5x+ faster to train. Give it a try!
	presets='high'     : Strong accuracy with fast inference speed.
	presets='high_v150': New in v1.5: Better quality than 'high' and 5x+ faster to train. Give it a try!


Beginning AutoGluon training ... Time limit = 300s


...

	0.0s	 = Training   runtime


	0.0s	 = Validation runtime


AutoGluon training complete, total runtime = 60.25s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 22800.4 rows/s (2501 batch size)


TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/Users/jnanadeep/Desktop/airbnb-price-prediction/notebooks/AutogluonModels/ag-20260708_164325")


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.415002,-0.422849,root_mean_squared_error,0.405197,0.109691,48.848986,0.000852,0.000101,0.002228,2,True,4
1,LightGBM,-0.416819,-0.428124,root_mean_squared_error,0.085512,0.036607,8.540252,0.085512,0.036607,8.540252,1,True,1
2,XGBoost,-0.422901,-0.428130,root_mean_squared_error,0.207239,0.052938,11.293385,0.207239,0.052938,11.293385,1,True,2
3,NeuralNetTorch,-0.471485,-0.471428,root_mean_squared_error,0.111594,0.020045,29.013121,0.111594,0.020045,29.013121,1,True,3


## Step 9: Evaluate the GPU-neural-net run and compare all three approaches

In [10]:
gpu_preds_log = predictor_gpu.predict(test_df.drop(columns=["price_log"]))
gpu_metrics = print_metrics_log_target("AutoGluon (GBM+XGB+NN_TORCH, NN on MPS GPU)", y_test_log, gpu_preds_log)

final_comparison = pd.DataFrame({
    "AutoGluon (unrestricted, no NN made the cut)": autogluon_metrics,
    "AutoGluon (GBM+XGB+NN_TORCH, NN on MPS GPU)": gpu_metrics,
    "Manual XGBoost (categorical + aspect sentiment)": xgb_metrics,
}).T[["mae", "rmse", "r2"]]
final_comparison


AutoGluon (GBM+XGB+NN_TORCH, NN on MPS GPU) | R^2 Score: 68.33% | MAE: $236.37 | RMSE: $482.64


,mae,rmse,r2
"AutoGluon (unrestricted, no NN made the cut)",234.396073,480.389527,0.686144
"AutoGluon (GBM+XGB+NN_TORCH, NN on MPS GPU)",236.374301,482.636290,0.683311
Manual XGBoost (categorical + aspect sentiment),245.138150,498.810873,0.659979
